### Data Reading

In [0]:
df = spark.table("bigmartsales.default.big_mart_sales")
from pyspark.sql.functions import *

In [0]:
df.display()

###Schema Definition

In [0]:
df.printSchema()

### DDL Schema

In [0]:
spark.read.load("bigmartsales.default.big_mart_sales")
spark.read.format("delta").load("bigmartsales.default.big_mart_sales")
df = spark.table("bigmartsales.default.big_mart_sales")


In [0]:
from pyspark.sql.functions import col

df = spark.table("bigmartsales.default.big_mart_sales") \
    .withColumn("Item_Weight", col("Item_Weight").cast("String"))


In [0]:
display(df)

### Struct Schema

In [0]:
from  pyspark.sql.functions import  *
from pyspark.sql.types import *
from pyspark.sql.functions import col


### SELECT


In [0]:
df_sel = df.select(col('Item_Identifier'), col('Item_Weight'), col('Item_Fat_Content')).display()


### ALIAS

In [0]:
df.select("Item_Identifier") \
  .withColumnRenamed("Item_Identifier", "Item_ID") \
  .display()


### FILTER


### Scenario - 1

In [0]:
df.filter(col('Item_Fat_Content') == 'Regular').display()

In [0]:
df.filter((col('Item_Type') == 'Soft Drinks') & (col('Item_Weight') < 10)).display()

### Scenario 3

In [0]:
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1','Tier 2'))).display()

### withColumnRenamed

In [0]:
df.withColumnRenamed('Item_Weight','Item_Wt').display()

### withColumn

In [0]:
df.withColumn('flag',lit("new")).display()

In [0]:
df.withColumn('multiply',col('Item_Weight') * col('Item_MRP')).display()

### Scenario - 2

In [0]:
df.withColumn('Item_Fat_content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
    .withColumn('Item_Fat_content',regexp_replace(col('Item_Fat_content'),"Low Fat","LF")).display()

### Type Casting

In [0]:
df = df.withColumn('Item_weight',col('Item_Weight').cast(StringType()))


### Sort

### Scenario - 1

In [0]:
df.sort(col('Item_weight').desc()).display()

### Scenario - 2

In [0]:
df.sort(col('Item_Visibility').asc()).display()

### Scenario - 3

In [0]:
df.sort(['Item_weight','Item_Visibility'],ascending=[0,0]).display()

### Scenario - 4

In [0]:
df.sort(['Item_weight','Item_Visibility'],ascending=[0,1]).display()

### Limit

In [0]:
df.limit(10).display()

### DROP

### Scenario - 2

In [0]:
df.drop('Item_Visibility','Item_Type').display()

### Drop_Duplicates

In [0]:
df.dropDuplicates().display()

### Scenario - 2

In [0]:
df.drop_duplicates(subset=['Item_Type']).display()

In [0]:
df.distinct().display()


### Union and Union BY Name

In [0]:
data1 = [('1','kad'),
         ('2','kads')]

schema1 = 'id string, name string'

df1 = spark.createDataFrame(data=data1, schema=schema1)

data1 = [('1','rahul'),
         ('2','jas')]
schema2 = 'id string, name string'

df2 = spark.createDataFrame(data=data1, schema=schema2)
df1.union(df2).display()

In [0]:
data1 = [('kad','1'),
         ('kads','2')]

schema1 = 'name string,id string'

df1 = spark.createDataFrame(data=data1, schema=schema1)

df1.display()

In [0]:
df1.unionByName(df2).display()

### String function

In [0]:
from pyspark.sql.functions import *

In [0]:
df.select(
    upper(col("Item_Type")).alias("Upper_Item_Type")
).display()


In [0]:
df.select(
    lower(col("Item_Type")).alias("Lower_Item_Type")
).display()

### Date Functions

In [0]:
df = df.withColumn('curr_date',current_date())
df.display()

### Date_Add()

In [0]:
df = df.withColumn('Week_after',date_add('curr_date',7))
df.display()

### Date_Sub()

In [0]:
df.withColumn('week_before',date_sub('curr_date',7)).display()

### DateDiff

In [0]:
df = df.withColumn('datediff',datediff('Week_after','curr_date'))
df.display()

### Date Format

In [0]:
df = df.withColumn(
    'curr_date',
    date_format(
        to_date(col('curr_date'), 'MM-dd-yyyy'),
        'dd-MM-yyyy'
    )
)
df.display()


### Dropping Null

In [0]:
df.dropna('all').display()


In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=['Outlet_Size']).display()

### Filling Nulls

In [0]:
df.fillna('NotAvailable', subset=[
    'Item_Fat_Content',
    'Item_Type',
    'Outlet_Size',
    'Outlet_Location_Type',
    'Outlet_Type'
]).display()


In [0]:
df.fillna(0, subset=[
    'Item_Weight',
    'Item_Visibility',
    'Item_MRP',
    'Item_Outlet_Sales'
]).display()


### Split and Indexing

In [0]:
df.withColumn(
    'Outlet_Type',
    split(
        col('Outlet_Type'),
        ' '
    )[1]
).display()


### Explode

In [0]:
df_exp = df.withColumn(
    'Outlet_Type',
    split(
        col('Outlet_Type'),
        ' '
    )
)

df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

### Array_Contains

In [0]:
df_exp.withColumn('Type1_flag',array_contains('Outlet_Type','Type1')).display()

### Group By

### Scenario 1 

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP')).display()



In [0]:
df.groupBy('Item_Type').agg(avg('Item_MRP')).display()

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP')).display()


In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP'),avg('Item_MRP').alias('Avg_MRP')).display()


### Collect List

In [0]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4')]

schema = 'user string, book string'

df_new = spark.createDataFrame(data,schema)

df_new.display()

In [0]:
from pyspark.sql.functions import collect_list

df_new.groupBy('user').agg(
    collect_list('book')
).display()

### PIVOT


In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

### When Otherwise

In [0]:
df=df.withColumn('veg_flag',when(col('Item_Type') == 'Meat','Non-Veg').otherwise('Veg'))
display(df)

In [0]:


df_new_when = df.withColumn(
    'veg_exp_flag',
    when(
        (col('veg_flag') == 'Veg') & (col('Item_MRP') < 100),
        'veg_Inexpensive'
    ).when(
        (col('veg_flag') == 'Veg') & (col('Item_MRP') > 100),
        'veg_Expensive'
    ).otherwise('Non-Veg')
)
display(df_new_when)

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','gauri','d02'),
          ('3','gaure','d02'),
          ('4','gauree','d04')]

schemaj1 = 'emp_id string, emp_name string, dept_id string'

df1 = spark.createDataFrame(dataj1,schemaj1)

display(df1)
dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','IT')]

schemaj2 = 'dept_id string, department string'

df2 = spark.createDataFrame(dataj2,schemaj2)

display(df2)

          

### Inner Join

In [0]:
df1.join(df2,df1.dept_id == df2.dept_id,'inner').display()

### Left Join

In [0]:
df1.join(df2,df1.dept_id == df2.dept_id,'left').display()

### Right Join

In [0]:
df1.join(df2,df1.dept_id == df2.dept_id,'right').display()

### Anti Join

In [0]:
df1.join(df2,df1.dept_id == df2.dept_id,'anti').display()

### Row Number

In [0]:
from pyspark.sql.window import *

In [0]:
df.withColumn('rowCol',row_number().over(Window.orderBy('Item_Identifier'))).display()

In [0]:
df.withColumn('rank',rank().over(Window.orderBy('Item_Identifier'))).display()

### Rank vs Dense Rank

In [0]:

 df.withColumn('rank', rank().over(Window.orderBy(col('Item_Identifier').desc())))\
     .withColumn('dense_rank', dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()


### Cumulative Sum

In [0]:
df.withColumn('cumsum', sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
df.withColumn('total',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()

### User Defined Function


In [0]:
def my_func(x):
    return x*x

In [0]:
my_udf = udf(my_func)

In [0]:
df.withColumn('mynewcol',my_udf('Item_MRP')).display()

### Data Writing

In [0]:
df.write.mode("overwrite").saveAsTable("my_table_name")

In [0]:
df.write.mode("append").saveAsTable("my_table_name")

#### Error

In [0]:
df.write.mode("error").saveAsTable("my_table_name")

### Ignore

In [0]:
df.write.mode("ignore").saveAsTable("my_table_name")

In [0]:
df.write.format('delta').mode("ignore").saveAsTable("my_table_name1")

### SparkSQL


In [0]:
df.createTempView('my_view')

In [0]:
%sql

select * from my_view where Item_Fat_Content = 'Lf'

In [0]:
df_sql = spark.sql("select * from my_view")
